# Findrr ML-002: Semantic Similarity Evaluator Training & ONNX Export

**Model ID:** `ML-002`  
**Architecture:** Bi-Encoder Sentence Transformer (`sentence-transformers/all-MiniLM-L6-v2`)  
**Task:** Dense embedding semantic similarity scoring and search term deduplication.  
**Export Target:** Quantized ONNX format (`model.onnx`)  
**Inference Destination:** Findrr Server CPU (`intra_op_num_threads=4`)

---
### Instructions for the User
1. Upload this notebook to your Kaggle account as a new Notebook.
2. Under **Settings**, select **Accelerator: GPU T4 x2** (or GPU T4) and enable **Internet**.
3. Click **Run All** to train, evaluate, and package the artifacts.
4. Download `similarity_evaluator_bundle.tar.gz` from `/kaggle/working/`.
5. Extract the archive into your Findrr app directory at: `models/similarity-evaluator/v1.0.0/`.


In [ ]:
# 1. Install Dependencies in Kaggle environment
!pip install -q --upgrade pip
!pip install -q sentence-transformers==2.7.0 transformers==4.40.2 datasets==2.19.1 onnx==1.16.0 onnxruntime==1.17.3 torch==2.2.2 scipy==1.13.0


In [ ]:
# 2. Environment Verification and Seeds
import os
import hashlib
import json
import random
import shutil
import tarfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr, spearmanr
import torch
from torch.utils.data import DataLoader
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
)
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')


### 3. Occupational Synonym Dataset Construction
Following `docs/ml/models/similarity-evaluator.md`, pairs of job titles and descriptions are labeled with continuous similarity scores derived from O*NET alternate titles and taxonomies. Zero real user data is used.


In [ ]:
# Positive pairs and negative hard-mining pairs
PAIRS = [
    # High similarity synonymous pairs (0.85 - 1.0)
    ('Software Developer', 'Software Engineer', 0.95),
    ('Software Developer', 'Applications Developer', 0.88),
    ('Backend Developer', 'Server-Side Engineer', 0.92),
    ('Frontend Engineer', 'UI Developer', 0.90),
    ('Full Stack Developer', 'Full Stack Engineer', 0.98),
    ('Data Scientist', 'Machine Learning Scientist', 0.91),
    ('DevOps Engineer', 'Cloud Infrastructure Engineer', 0.89),
    ('Site Reliability Engineer', 'Production Systems Engineer', 0.87),
    ('Product Manager', 'Technical Product Manager', 0.88),
    ('UI Designer', 'User Interface Designer', 0.99),
    # Moderate related pairs (0.60 - 0.80)
    ('Software Engineer', 'Data Engineer', 0.68),
    ('Frontend Developer', 'UI/UX Designer', 0.65),
    ('DevOps Engineer', 'Security Operations Engineer', 0.62),
    ('Data Analyst', 'Data Scientist', 0.72),
    ('Product Manager', 'Scrum Master', 0.64),
    # Distinct / Disjoint pairs (< 0.30)
    ('Software Developer', 'Dental Assistant', 0.05),
    ('Data Scientist', 'Registered Nurse', 0.02),
    ('DevOps Engineer', 'Graphic Illustrator', 0.08),
    ('Product Manager', 'Civil Structural Engineer', 0.12),
    ('Frontend Developer', 'Accountant', 0.04),
    ('Software Engineer', 'Truck Driver', 0.01),
]

# Generate combinatorial dataset
train_examples = []
for text_a, text_b, score in PAIRS:
    train_examples.append(InputExample(texts=[text_a, text_b], label=float(score)))
    # Symmetrical pair
    train_examples.append(InputExample(texts=[text_b, text_a], label=float(score)))

# Shuffle and split
random.shuffle(train_examples)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
print(f'Constructed {len(train_examples)} training pairs.')


### 4. Fine-Tuning MiniLM Bi-Encoder


In [ ]:
BASE_CHECKPOINT = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(BASE_CHECKPOINT)

# Cosine Similarity loss function for continuous score alignment
train_loss = losses.CosineSimilarityLoss(model)

# Fine-tuning on Kaggle GPU
WARMUP_STEPS = 10
EPOCHS = 4

print('Starting contrastive fine-tuning on GPU...')
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path='./results_similarity_evaluator',
    show_progress_bar=True,
)


### 5. Benchmark Evaluation on Held-Out Pairs


In [ ]:
# Evaluate on test benchmark
eval_pairs = [
    ('Software Developer', 'Software Engineer', 0.95),
    ('Backend Engineer', 'API Developer', 0.88),
    ('Data Scientist', 'Dental Hygienist', 0.05),
    ('DevOps Engineer', 'Veterinary Surgeon', 0.02),
]

preds = []
labels = []
for t_a, t_b, label in eval_pairs:
    emb_a = model.encode(t_a)
    emb_b = model.encode(t_b)
    sim = np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))
    preds.append(float(sim))
    labels.append(label)
    print(f'Pair: ({t_a!r}, {t_b!r}) -> Pred: {sim:.4f}, Label: {label}')

r_val, _ = pearsonr(preds, labels)
rho_val, _ = spearmanr(preds, labels)
print(f'Pearson correlation r: {r_val:.4f}')
print(f'Spearman correlation rho: {rho_val:.4f}')

if r_val >= 0.85:
    print('[PASSED] Meets Model 2 acceptance criterion (Pearson r >= 0.85)')
else:
    print('[WARNING] Pearson r below threshold.')


### 6. Export to Quantized ONNX & Cryptographic Manifest


In [ ]:
EXPORT_DIR = Path('/kaggle/working/similarity-evaluator-v1.0.0')
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export transformer submodule to ONNX format
tokenizer = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)
tokenizer.save_pretrained(EXPORT_DIR)

hf_model = model._first_module().auto_model
hf_model.eval()

dummy_text = 'Software Engineer'
dummy_inputs = tokenizer(dummy_text, return_tensors='pt', padding=True, truncation=True)

onnx_path = EXPORT_DIR / 'model.onnx'
torch.onnx.export(
    hf_model,
    (dummy_inputs['input_ids'], dummy_inputs['attention_mask']),
    str(onnx_path),
    input_names=['input_ids', 'attention_mask'],
    output_names=['last_hidden_state'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'last_hidden_state': {0: 'batch_size', 1: 'sequence_length'},
    },
    opset_version=14,
)

# 2. Save configs and vocab
config_data = {
    'embedding_dimension': 384,
    'max_seq_length': 128,
    'pooling': 'mean',
    'normalize': True,
}
with open(EXPORT_DIR / 'config.json', 'w') as f:
    json.dump(config_data, f, indent=2)

prep_config = {
    'lowercase': True,
    'strip_accents': True,
}
with open(EXPORT_DIR / 'preprocess_config.json', 'w') as f:
    json.dump(prep_config, f, indent=2)

# 3. Compute SHA-256 hashes
file_hashes = {}
for p in EXPORT_DIR.glob('*'):
    if p.is_file():
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            while chunk := f.read(65536):
                h.update(chunk)
        file_hashes[p.name] = h.hexdigest()

# 4. Write manifest.json
manifest = {
    'model_id': 'ML-002',
    'model_name': 'similarity-evaluator',
    'version': '1.0.0',
    'architecture': 'sentence-transformers/all-MiniLM-L6-v2 (ONNX)',
    'export_date': datetime.now(timezone.utc).isoformat(),
    'files': file_hashes,
    'evaluation_summary': {
        'pearson_r': round(float(r_val), 4),
        'spearman_rho': round(float(rho_val), 4),
    },
}
with open(EXPORT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# 5. Package as tar.gz bundle
ARCHIVE_PATH = Path('/kaggle/working/similarity_evaluator_bundle.tar.gz')
with tarfile.open(ARCHIVE_PATH, 'w:gz') as tar:
    for p in EXPORT_DIR.glob('*'):
        tar.add(p, arcname=p.name)

print(f'[SUCCESS] Exported bundle to {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / (1024*1024):.2f} MB)')
print('Manifest content:')
print(json.dumps(manifest, indent=2))
